In [ ]:
"""
Multi-window, lead-conditioned UNet downscaling: ECMWF S2S -> IMD 0.25 rain
anomaly over India, trained jointly across several DISJOINT forecast windows,
evaluated by rotating-year cross-validation.

WHY MULTI-WINDOW (the point of this rewrite)
--------------------------------------------
The single-window version reduced each init to one sample (3720 total) and
overfit hard -- and changing model size did nothing, which means the binding
constraint is SAMPLES, not capacity. Two facts drive the fix:

  * Adding all daily leads does NOT add real data. Day-20 and day-21 from one
    init share nearly identical predictor fields and nearly identical targets.
    43 daily leads from an init is ~3 effectively-independent samples, not 43.
    Worse, adjacent leads split across train/val is near-duplication = leak.

  * Adding DISJOINT windows DOES add data. Week-2, weeks 3-4 and weeks 5-6
    dynamics genuinely differ, and their valid-date ranges don't overlap, so
    no leak. One model correcting all three, told which window it is via an
    embedding channel, shares structure across leads -- the data-rich short
    leads regularise the data-poor long leads.

So: WINDOWS below are non-overlapping. Each (init, window) is one sample with
a window-id. Effective sample count ~3x. This is the lever that actually
moves val skill; base/dropout/weight-decay are second-order.

WHY ROTATING-YEAR CV
--------------------
20 years is few. A single 3-year val holdout is noisy -- one El Nino year in
val can dominate the estimate. Rotating leave-3-out CV over the non-test years
gives a stable mean +/- spread and uses every year for validation once.
Test years are held out of ALL folds, always.

Everything else carries over from the settled pipeline: anomaly space with
train-only climatology, strict mask, mask+lat/lon static channels,
coarse-input with in-model grid_sample upsampling, GroupNorm, masked loss.

USAGE
-----
  python s2s_unet_mw.py prepare        # archive -> cached arrays (slow, once)
  python s2s_unet_mw.py train          # rotating-year CV, then final model
  python s2s_unet_mw.py train --epochs 60 --batch 16 --base 24 --folds 5
"""

import argparse
import os
import time
from contextlib import contextmanager

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"

# DISJOINT windows: (name, first_lead_day, last_lead_day) inclusive.
# Non-overlap in valid dates is what keeps the extra samples honest.
WINDOWS = [
    ("week1", 1, 7),
    ("week2", 8, 14),
    ("week3", 15, 21),
    ("week4", 22, 28),
    ("week5", 29, 35),
    ("week6", 36, 42),
]

CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
TEST_YEARS_N = 3                # held out of every CV fold
CACHE = "unet_cache_mw.npz"
OUT_MAPS = "unet_skill_maps_mw.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


In [ ]:

# PREPARE  (numpy/xarray only; torch not needed here)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...), NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)




def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Build each window's (X, y, doy) then stack, tagging window id.
    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")

            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)

            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values

            X_list.append(Xa)
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list)          # (N, V, h, w)  N = n_init * n_window
    y = np.concatenate(y_list)          # (N, H, W)
    doy = np.concatenate(doy_list)      # (N,)
    wid = np.concatenate(wid_list)      # (N,)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        # strict mask: valid on every day, per window then intersect
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose {int(np.isfinite(y).any(axis=0).sum())})")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years (held out of all folds): {sorted(test_years)}")

    with stage("Caching"):
        # NOTE: climatology + standardisation are deferred to train time,
        # because they must be recomputed per CV FOLD (train-year stats only).
        # Caching raw windowed anomable inputs would bake in a fixed split.
        np.savez_compressed(
            cache_path,
            X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clat=clat, clon=clon, flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")
        print("    (raw windowed values cached; climatology/standardisation")
        print("     done per-fold at train time so each fold is leak-free)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING  (must run per fold: train-year stats only)
# ======================================================================

def anomalise_fold(X, y, doy, tr):
    """De-climatologise both sides using TRAIN-fold years only, then
    standardise predictors on train stats. Returns processed copies.

    This is the leakage-critical step. Doing it once globally would let the
    val/test years inform the climatology; doing it per fold keeps each
    fold's estimate honest.
    """
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, H, W)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, V, h, w)

    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]

    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


In [1]:
# ======================================================================
# MODEL + TRAIN  (torch, lazy import)
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"])
    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    # ---- coarse->fine grid_sample grid (coordinate-aware bilinear) ----
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box; raise COARSE_PAD"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            layers = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                      nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                layers.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*layers)

        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        """Coarse predictors + window-id -> fine anomaly field.

        The window id is embedded and broadcast as an extra coarse-input
        channel, so one model corrects all windows and shares structure
        across leads. Everything else is the settled downscaling UNet:
        coarse encoder -> grid_sample to fine -> static channels -> small
        UNet refinement.
        """
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], dim=1)
            c = self.enc_c2(self.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def masked_mse(pred, target, m):
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        """Per-cell skill (vs zero-anomaly clim) and ACC over masked cells."""
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6                       # guard degenerate cells (-inf fix)
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs, tag):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss = masked_mse(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(masked_mse(model(xb, wb, samp, stat), yb, mb)) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa)
        widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                p = model(Xt[j].to(dev), widt[j].to(dev), samp, stat)
                out.append(p.cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV over non-test years ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    folds = args.folds
    # contiguous blocks of years as val, rest as train
    blocks = np.array_split(nontest_years, folds)

    with stage(f"Rotating-year CV: {folds} folds over {len(nontest_years)} years"):
        fold_skl, fold_acc = [], []
        for fi, val_years in enumerate(blocks):
            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test

            # per-fold anomalisation (train years of THIS fold only)
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs, f"fold{fi}")
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]      # mask from RAW y
            skill, acc = skill_acc(p, t, fin)
            ms = float(np.nanmean(skill[mask]))
            ma = float(np.nanmean(acc[mask]))
            fold_skl.append(ms)
            fold_acc.append(ma)
            print(f"    fold {fi} val {sorted(val_years)}: "
                  f"skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep | vloss {vloss:.3f}",
                  flush=True)

        print(f"\n    CV skill {np.mean(fold_skl):+.3f} +/- {np.std(fold_skl):.3f}"
              f"   CV ACC {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

    # ---------- final model: train on ALL non-test, evaluate on test ----------
    with stage("Final model on all non-test years -> test"):
        tr_i = ~is_test
        # small val slice just for early stopping (last 2 non-test years)
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = tr_i & ~va_i

        Xa, ya, clim_y = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0],
                                  Xa, ya, args.epochs, "final")

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        # overall + per-window test skill
        import xarray as xr
        wid_te = wid[is_test]
        data_vars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sub_m = wid_te == w
            if sub_m.sum() == 0:
                continue
            sk, ac = skill_acc(p[sub_m], t[sub_m], fin[sub_m])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.3f} | "
                  f"ACC {np.nanmean(ac[mask]):.3f} | "
                  f"{100*np.nanmean(sk[mask] > 0):.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.attrs["note"] = ("anomaly-space skill = 1 - rmse/rmse_clim (clim = "
                             "zero anomaly); NaN outside IMD mask / degenerate cells")
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt). Per-window skill maps; open in Panoply.")


# ======================================================================

# Jupyter Notebook equivalent of command-line args
class Args:
    cmd = 'train' # Change to 'train' to train the model
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()

if args.cmd == 'prepare':
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
    imd_ds = ds_imd      # noqa: F821
    prepare(ecmwf_ds, imd_ds, CACHE)
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f'run `prepare` first ({CACHE} missing)')
    build_and_run(args, CACHE, OUT_MAPS)


NameError: name 'os' is not defined

In [1]:
import os
import time

filename = "unet_cache_mw.npz"
if os.path.exists(filename):
    print(f"Size: {os.path.getsize(filename) / 1e9:.2f} GB")
    print(f"Last modified: {time.ctime(os.path.getmtime(filename))}")
else:
    print("File not created yet.")


Size: 1.06 GB
Last modified: Tue Jul 21 14:30:58 2026
